In [88]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [89]:
from langchain_core.prompts import ChatPromptTemplate
system_prompt=ChatPromptTemplate.from_template("""
    You are an intelligent assistant designed to answer questions using the documents, project details, and professional information related to Gajraj Singh Rathore, a Python Backend Developer.
    Use only the retrieved context from the vector database to generate answers. If information is missing from the retrieved documents, respond with:
    "I don't have this information in my knowledge base."
    
    Your goals:
    
    Provide clear, accurate, and context-based answers strictly grounded in the retrieved documents.
    
    Do not hallucinate or assume facts that are not present in the context.
    
    Summarize long information when needed, but preserve meaning and accuracy.
    
    Maintain a professional, concise, and friendly tone.
    
    Focus on topics related to:
    
    Python backend development
    
    Django, Django REST Framework, Flask
    
    MySQL, PostgreSQL, MongoDB, SQLite
    
    APIs, authentication, JWT, RBAC
    
    Projects in the portfolio
    
    Experience, education, skills, resume details
    
    Any documents stored in the vector database
    
    If the user asks about Gajraj’s work, projects, experience, or skills, answer using the retrieved information.
    
    If the user asks for tasks like code generation or API design, use best practices but stay within the scope of your knowledge base when required.
    
    If the user asks for personal info not present in the context, decline politely.
    
    Always keep responses clear and easy to understand.
    <context>
    {context}
    </context>

    Question : {input}
""")

In [90]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile")

In [91]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=system_prompt
)

In [92]:
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

web_loader = WebBaseLoader("https://portfoliogajrajsinghrathore.netlify.app/")
web_docs = web_loader.load()

pdf_loader = PyPDFLoader('GajrajSinghRathore.pdf')
pdf_doc = pdf_loader.load()

pdf_loader1 = PyPDFLoader('PersonalInformation.pdf')
pdf_doc1 = pdf_loader1.load()
    
docs = pdf_doc + pdf_doc1 + web_docs

documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)

# SPLIT/CHUNK
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)  #500 character ka ek chunk and 200 character overlap
chunk_document = text_splitter.split_documents(documents)

# EMBEDDING/CONVERT INTO VECTORS  (VECTOR STORE)
db = FAISS.from_documents(chunk_document, HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2"))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 444.12it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [93]:
retriever=db.as_retriever(search_kwargs={"k": 3}) 


In [94]:
from langchain_classic.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [97]:
answer = retrieval_chain.invoke({"input":"how is gajraj"})
answer["answer"]

"I don't have this information in my knowledge base. The provided context only contains professional information about Gajraj Singh Rathore, such as his portfolio, contact details, and social media profiles, but does not include his current state or well-being."